<a href="https://colab.research.google.com/github/tayyba6/ML_Internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**One row represents one pseudonymized content page at a March 2026 decision point.** The page-level row is built from the daily performance history available up to the end of March 2026.

For the feature window, I use the **90 days before and including 2026-03-31**. This gives a consistent historical window for measuring search visibility, clicks, position, and content lifecycle signals before making a ranking decision.

My lane is **Refresh / Content Opportunity Scoring**. The practical decision is which pages should be reviewed first for refresh or related content action. I use observed negative movement as a **proxy/evaluation signal**, not as proof that a refresh will cause recovery.

The warehouse source is `fact_content_daily_performance`, with `dim_content` used only when content-level metadata is needed. The warehouse daily table has the grain of one report date × client × content item, so I aggregate it to my page-level decision grain rather than treating each daily row as an independent page.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features
I use five features for the initial ranking frame:
1. **`content_age_days`** — age of the content at the decision point.
2. **`days_since_last_update`** — freshness/lifecycle signal available before the decision.
3. **`impressions`** — search visibility/exposure observed during the feature window.
4. **`avg_position`** — average search position observed during the feature window.
5. **`ctr`** — click-through rate calculated from observed impressions and clicks during the feature window.

### Label / proxy
The evaluation proxy is **negative movement**, represented by the warehouse's observed trend signal. This is a proxy for pages that may deserve review; it is not a causal label saying that a refresh is required or guaranteed to work.

### Context
The following fields provide context rather than model features:
* `client_hash_id`
* `content_hash_id`
* `report_date`
* data-availability indicators
* content metadata used only to understand the slice

### Excluded
I deliberately exclude **`trend_pct` and `trend_direction` from the honest feature set** because they describe movement that is also used to define the outcome/proxy. Including them would allow information about the answer to enter the features and would create leakage.
I also exclude any product decision fields such as `health_score`, `priority_score`, `action_type`, or refresh flags. These are product decisions rather than independent observable signals and should not become model features.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [2]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_Token")

if HF_TOKEN:
    print("HF_TOKEN loaded successfully.")
else:
    print("HF_TOKEN not found.")

HF_TOKEN loaded successfully.


In [3]:
import duckdb
import pandas as pd
import numpy as np

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    )
    """
)

print("Hugging Face authentication configured.")

Hugging Face authentication configured.


In [4]:
FACT = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/**/*.parquet"
)

test = con.sql(f"""
    SELECT *
    FROM read_parquet('{FACT}')
    WHERE month = '2026-03'
    LIMIT 5
""").df()

test

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


### Query 1 — Grain verification

In [5]:
q1 = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(
            DISTINCT
            CAST(report_date AS VARCHAR)
            || '|' ||
            client_hash_id
            || '|' ||
            content_hash_id
        ) AS distinct_grain_keys
    FROM read_parquet('{FACT}')
    WHERE month = '2026-03'
""").df()

q1

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,distinct_grain_keys
0,9841378,9841378


The March 2026 slice contains 9,841,378 rows and 9,841,378 distinct `report_date + client_hash_id + content_hash_id` keys. Because these counts are equal, the rows support the expected grain of one daily observation for one client-content pair.
This verifies the warehouse fact-table grain rather than assuming it from the schema.

#Query 2 — Row count + date span

In [6]:
q2 = con.sql(f"""
    SELECT
        COUNT(*) AS march_rows,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM read_parquet('{FACT}')
    WHERE month = '2026-03'
""").df()

q2

,march_rows,first_date,last_date
0,9841378,2026-03-01,2026-03-31


The March 2026 slice contains 9,841,378 daily fact rows, covering the full observed date range from 2026-03-01 through 2026-03-31.
I use March 2026 as the development month because it is a mid-panel month rather than the final June outcome month. This keeps development separate from the final month that is reserved as a sealed outcome window.

#Query 3 — Availability

In [7]:
q3 = con.sql(f"""
    SELECT
        COUNT(*) AS march_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows
    FROM read_parquet('{FACT}')
    WHERE month = '2026-03'
""").df()

q3

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,march_rows,gsc_available_rows
0,9841378,3611061


Of the 9,841,378 March 2026 fact rows, 3,611,061 rows have `gsc_data_available IS TRUE`, which is approximately 36.69% of the March slice.
I use the explicit availability flag rather than treating missing search data as zero. This matters because a missing data source and a measured zero are different states.
For this Search Intelligence lane, I therefore restrict search-based feature calculations to observations where GSC data is confirmed available.

In [8]:
schema = con.sql(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{FACT}')
""").df()

schema[["column_name", "column_type"]]

,column_name,column_type
0,report_date,DATE
1,client_hash_id,VARCHAR
2,content_hash_id,VARCHAR
3,client_has_gsc,BOOLEAN
4,client_has_ga4,BOOLEAN
5,gsc_data_available,BOOLEAN
6,ga4_data_available,BOOLEAN
7,gsc_impressions,BIGINT
8,gsc_clicks,BIGINT
9,gsc_sum_position,BIGINT


In [9]:
feature_sql = f"""
WITH march AS (
    SELECT *
    FROM read_parquet('{FACT}')
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
      AND gsc_data_available IS TRUE
)

SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_impressions) AS gsc_impressions_30d,

    SUM(gsc_clicks) AS gsc_clicks_30d,

    CASE
        WHEN SUM(gsc_impressions) > 0
        THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
        ELSE NULL
    END AS gsc_ctr_30d,

    AVG(gsc_avg_position) AS gsc_avg_position_30d,

    SUM(
        CASE
            WHEN ga4_data_available IS TRUE
            THEN ga4_sessions
            ELSE NULL
        END
    ) AS ga4_sessions_30d

FROM march

GROUP BY
    client_hash_id,
    content_hash_id
"""

features = con.sql(feature_sql).df()

features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,gsc_impressions_30d,gsc_clicks_30d,gsc_ctr_30d,gsc_avg_position_30d,ga4_sessions_30d
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,77.0,0.0,0.000000,4.074107,NaN
1,client_62f4a7e64f5e0096,content_c03ecafd4c999f15,10849.0,22.0,0.002028,8.240351,NaN
2,client_62f4a7e64f5e0096,content_e689bc511192751a,61.0,0.0,0.000000,6.015432,NaN
3,client_62f4a7e64f5e0096,content_7dbc094b799e05a4,705.0,1.0,0.001418,5.956862,NaN
4,client_62f4a7e64f5e0096,content_40b10da45f4c1cb5,50.0,0.0,0.000000,12.977513,NaN


### Feature availability

The five features are constructed from observations available within the March 2026 historical window. GSC-based features are restricted to rows where `gsc_data_available IS TRUE`.

`ga4_sessions_30d` remains missing when GA4 is unavailable. I do not replace missing analytics with zero because unavailable tracking does not mean that zero sessions occurred.
Each feature is therefore based on information that could have been observed by the March 31 decision point.

In [10]:
print("Feature frame shape:", features.shape)
print()
print("Missing values:")
print(features.isna().sum())

Feature frame shape: (176738, 7)

Missing values:
client_hash_id               0
content_hash_id              0
gsc_impressions_30d          0
gsc_clicks_30d               0
gsc_ctr_30d                  0
gsc_avg_position_30d         0
ga4_sessions_30d        112882
dtype: int64


### Feature-quality check

The resulting feature frame contains 176,738 client-content observations and five candidate features.

The four GSC-derived features have no missing values after filtering to
`gsc_data_available IS TRUE`. `ga4_sessions_30d` is missing for 112,882
observations, or approximately 63.87% of the feature frame.

I keep these GA4 values missing rather than converting them to zero because
unavailable analytics tracking does not imply zero sessions. This is an
important limitation of the feature set and means that models using this
feature must handle missing values explicitly.

In [11]:
april_columns = con.sql(f"""
    SELECT *
    FROM read_parquet('{FACT}')
    WHERE month = '2026-04'
    LIMIT 5
""").df()

april_columns

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-04-01,client_62f4a7e64f5e0096,content_143987cfdeaba4c0,True,False,True,<NA>,9,0,493,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-04
1,2026-04-01,client_62f4a7e64f5e0096,content_13a8105125458098,True,False,True,<NA>,1,0,9,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-04
2,2026-04-01,client_62f4a7e64f5e0096,content_6a887d56ab6c8362,True,False,False,<NA>,0,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-04
3,2026-04-01,client_62f4a7e64f5e0096,content_e2bd76be7eed690d,True,False,True,<NA>,1,0,7,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-04
4,2026-04-01,client_62f4a7e64f5e0096,content_ddbfb1907979759a,True,False,False,<NA>,0,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-04


#Step 1 — Build the March → April outcome

In [12]:
outcome_sql = f"""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS march_clicks
    FROM read_parquet('{FACT}')
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
      AND gsc_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
),

april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS april_clicks
    FROM read_parquet('{FACT}')
    WHERE report_date BETWEEN DATE '2026-04-01' AND DATE '2026-04-30'
      AND gsc_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    m.client_hash_id,
    m.content_hash_id,
    m.march_clicks,
    a.april_clicks,

    CASE
        WHEN a.april_clicks < m.march_clicks THEN 1
        ELSE 0
    END AS decline_label

FROM march m
INNER JOIN april a
    ON m.client_hash_id = a.client_hash_id
   AND m.content_hash_id = a.content_hash_id

WHERE a.april_clicks IS NOT NULL
"""

outcomes = con.sql(outcome_sql).df()

outcomes.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,march_clicks,april_clicks,decline_label
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,7.0,8.0,0
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,0.0,2.0,0
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,6.0,4.0,1
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,13.0,8.0,1
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,1.0,0.0,1


In [13]:
print("Outcome frame shape:", outcomes.shape)
print()
print(outcomes["decline_label"].value_counts())

Outcome frame shape: (158549, 5)

decline_label
0    114244
1     44305
Name: count, dtype: int64


#Step 2 — Combine features and label

In [14]:
model_df = features.merge(
    outcomes,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

print("Model dataframe shape:", model_df.shape)
model_df.head()

Model dataframe shape: (158549, 10)


,client_hash_id,content_hash_id,gsc_impressions_30d,gsc_clicks_30d,gsc_ctr_30d,gsc_avg_position_30d,ga4_sessions_30d,march_clicks,april_clicks,decline_label
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,77.0,0.0,0.000000,4.074107,NaN,0.0,0.0,0
1,client_62f4a7e64f5e0096,content_c03ecafd4c999f15,10849.0,22.0,0.002028,8.240351,NaN,22.0,23.0,0
2,client_62f4a7e64f5e0096,content_e689bc511192751a,61.0,0.0,0.000000,6.015432,NaN,0.0,1.0,0
3,client_62f4a7e64f5e0096,content_7dbc094b799e05a4,705.0,1.0,0.001418,5.956862,NaN,1.0,0.0,1
4,client_62f4a7e64f5e0096,content_40b10da45f4c1cb5,50.0,0.0,0.000000,12.977513,NaN,0.0,0.0,0


In [15]:
print(model_df["decline_label"].value_counts(normalize=True))

decline_label
0    0.72056
1    0.27944
Name: proportion, dtype: float64


#Step 3 — Create the deliberate leakage feature

In [16]:
LEAKY_FEATURES = [
    "gsc_impressions_30d",
    "gsc_clicks_30d",
    "gsc_ctr_30d",
    "gsc_avg_position_30d",
    "ga4_sessions_30d",
    "april_clicks",       # DELIBERATE LEAK
]
HONEST_FEATURES = [
    "gsc_impressions_30d",
    "gsc_clicks_30d",
    "gsc_ctr_30d",
    "gsc_avg_position_30d",
    "ga4_sessions_30d",
]

#Step 4 — Train the deliberately leaky model

In [17]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

X = model_df[LEAKY_FEATURES]
y = model_df["decline_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

leaky_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("tree", DecisionTreeClassifier(
        max_depth=3,
        random_state=42
    ))
])

leaky_model.fit(X_train, y_train)

leaky_pred = leaky_model.predict(X_test)

leaky_score = accuracy_score(y_test, leaky_pred)

print("Leaky accuracy:", leaky_score)

Leaky accuracy: 0.8989908546199937


#Step 5 — Train the honest model

In [18]:
X = model_df[HONEST_FEATURES]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

honest_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("tree", DecisionTreeClassifier(
        max_depth=3,
        random_state=42
    ))
])

honest_model.fit(X_train, y_train)

honest_pred = honest_model.predict(X_test)

honest_score = accuracy_score(y_test, honest_pred)

print("Honest accuracy:", honest_score)

Honest accuracy: 0.8514033427940713


### Deliberate leakage experiment

I deliberately added `april_clicks` to the feature set. This is a leaked feature because the label itself is defined using April clicks:

`decline_label = 1` when April clicks are lower than March clicks.

The leaky model achieved the score shown above, while the honest model was trained without April information.

The inflated leaky result does not demonstrate useful predictive performance. It demonstrates target leakage: information from the outcome period was allowed to enter the model inputs.

I therefore removed `april_clicks` and retained the honest feature set for future modeling.

The honest score is the result that should be used for comparison in later work.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## 4. Data limits

### Named limitation

A major limitation of this slice is uneven analytics availability. Although
GSC availability is explicitly filtered for the search-based features, GA4
sessions remain unavailable for 112,882 of the 176,738 feature rows
(approximately 63.87%).

Therefore, analytics-based signals are not uniformly observed across the
entire slice. Missing GA4 data should not be interpreted as zero traffic, and
any later model using this feature must account for missingness.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.